# Lumis 11: SCIN phone photos, split for model v2

Model v1 reached 65.5% on DermNet's clinical photos but only 37.7% on SCIN's self-taken phone photos (notebook 10a).
Model v2 also learns from phone photos. This notebook prepares them, and **freezes the split before any training**.

- Cases: every SCIN case whose main dermatologist label (at least half of the combined weight) is one of the five
  Lumis conditions. Contact dermatitis is not eczema (same rule as notebook 10a).
- Photos: **all** photos of each case (up to 3: close-up, angle, distance) go to training; evaluation uses one photo per
  case, the close-up, because that is what a Lumis user takes.
- Split by **case** (one case = one person), stratified by condition: 60% train, 20% validation, 20% test.
  Identical photos found in more than one case are kept in the same split.
- The **test** part is used once, in notebook 14.

Licence: SCIN Data Use License (attribution; never try to identify contributors). Cite Ward et al., arXiv:2402.18545 (2024).

CPU runtime is fine. About 15 to 20 minutes, mostly downloading.

In [ ]:
MIN_AGREEMENT = 0.5
SPLIT = {'train': 0.6, 'val': 0.2, 'test': 0.2}
SEED = 42

In [ ]:
!pip install -q google-cloud-storage
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'
OUT_CSV = f'{BASE}/data/scin_split_v1.csv'
OUT_ZIP = f'{BASE}/data/scin_images.zip'
import os
if os.path.exists(OUT_CSV):
    raise RuntimeError(f'{OUT_CSV} already exists: the split is frozen. Delete it only if you mean to redo the split before any training.')

In [ ]:
import os, io, ast, json, time, hashlib, zipfile
import numpy as np, pandas as pd
from google.cloud import storage

BUCKET = 'dx-scin-public-data'
try:
    bucket = storage.Client.create_anonymous_client().bucket(BUCKET)
    bucket.blob('dataset/scin_cases.csv').download_as_bytes(start=0, end=10)
except Exception as e:
    print('Anonymous access failed, signing in instead (a popup will ask you to allow access):', type(e).__name__)
    from google.colab import auth
    auth.authenticate_user()
    bucket = storage.Client(project='dx-scin-public').bucket(BUCKET)

def fetch(path):
    return bucket.blob(path).download_as_bytes()

cases = pd.read_csv(io.BytesIO(fetch('dataset/scin_cases.csv')), dtype={'case_id': str})
labels = pd.read_csv(io.BytesIO(fetch('dataset/scin_labels.csv')), dtype={'case_id': str})
scin = cases.merge(labels, on='case_id')
print(len(scin), 'SCIN cases')

In [ ]:
def parse(x):
    if isinstance(x, str) and x.strip():
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return {}
    return {}

def to_lumis(name):
    n = name.lower()
    if 'scabies' in n:
        return 'scabies'
    if 'psoria' in n and 'parapsoria' not in n:
        return 'psoriasis'
    if 'tinea' in n or 'dermatophyt' in n or 'ringworm' in n:
        return 'tinea'
    if n.startswith('acne') and 'rosacea' not in n and 'keloid' not in n:
        return 'acne'
    if ('eczema' in n or 'atopic' in n) and 'contact' not in n:
        return 'eczema'
    return None

def main_label(w):
    if not w:
        return None, 0.0
    name, weight = max(w.items(), key=lambda kv: kv[1])
    return to_lumis(name), weight

def fitz(r):
    vals = [r.get(f'dermatologist_fitzpatrick_skin_type_label_{i}') for i in (1, 2, 3)]
    vals = [v for v in vals if isinstance(v, str) and v.startswith('FST')]
    if not vals:
        return None
    v = int(round(np.median([int(x[3]) for x in vals])))
    return 'I-II' if v <= 2 else ('III-IV' if v <= 4 else 'V-VI')

scin['weights'] = scin.weighted_skin_condition_label.map(parse)
scin[['lumis_class', 'agreement']] = scin.weights.apply(lambda w: pd.Series(main_label(w)))
scin['tone'] = scin.apply(fitz, axis=1)

chosen = scin[scin.lumis_class.notna() & (scin.agreement >= MIN_AGREEMENT)].copy()
print(len(chosen), 'cases')
print(pd.crosstab(chosen.lumis_class, chosen.tone.fillna('unknown'), margins=True))

## 1. Download every photo of those cases

In [ ]:
LOCAL = '/content/scin'
os.makedirs(LOCAL, exist_ok=True)
rows = []
t0 = time.time()
for _, r in chosen.iterrows():
    for i in (1, 2, 3):
        p = r.get(f'image_{i}_path')
        if not isinstance(p, str):
            continue
        ext = os.path.splitext(p)[1] or '.png'
        name = f'scin/{r.case_id}_{i}{ext}'
        local = f'/content/{name}'
        if not os.path.exists(local):
            try:
                open(local, 'wb').write(fetch(p))
            except Exception as e:
                print('skipped', p, type(e).__name__)
                continue
        rows.append({'path': name, 'case_id': r.case_id, 'image_no': i, 'shot_type': r.get(f'image_{i}_shot_type'),
                     'lumis_class': r.lumis_class, 'agreement': round(float(r.agreement), 3), 'tone': r.tone,
                     'md5': hashlib.md5(open(local, 'rb').read()).hexdigest()})
imgs = pd.DataFrame(rows).drop_duplicates(['case_id', 'md5'])     # same photo uploaded twice in one case
print(f'{len(imgs)} photos from {imgs.case_id.nunique()} cases in {time.time() - t0:.0f} s')

## 2. Evaluation photo per case, duplicates, and the split

The evaluation photo is the close-up if there is one, otherwise the first photo.

In [ ]:
def eval_photo(g):
    close = g[g.shot_type == 'CLOSE_UP']
    return (close if len(close) else g).sort_values('image_no').path.iloc[0]

eval_paths = set(imgs.groupby('case_id').apply(eval_photo))
imgs['eval_photo'] = imgs.path.isin(eval_paths)

# Cases sharing an identical photo are grouped so they land in the same split
parent = {c: c for c in imgs.case_id.unique()}
def find(c):
    while parent[c] != c:
        parent[c] = parent[parent[c]]
        c = parent[c]
    return c
for _, g in imgs.groupby('md5'):
    cs = g.case_id.unique()
    for c in cs[1:]:
        parent[find(c)] = find(cs[0])
imgs['group'] = imgs.case_id.map(find)
shared = imgs.groupby('md5').case_id.nunique()
print('Photos shared by more than one case:', int((shared > 1).sum()), '| case groups:', imgs.group.nunique(), 'from', imgs.case_id.nunique(), 'cases')

rng = np.random.default_rng(SEED)
groups = imgs.drop_duplicates('group')[['group', 'lumis_class']]
assign = {}
for cls, g in groups.groupby('lumis_class'):
    ids = rng.permutation(sorted(g.group.astype(str)))
    n = len(ids)
    n_test = max(1, int(round(n * SPLIT['test'])))
    n_val = max(1, int(round(n * SPLIT['val'])))
    for gid in ids[:n_test]:
        assign[gid] = 'test'
    for gid in ids[n_test:n_test + n_val]:
        assign[gid] = 'val'
    for gid in ids[n_test + n_val:]:
        assign[gid] = 'train'
imgs['split'] = imgs.group.map(assign)

print('\nCases per split:')
print(pd.crosstab(imgs.drop_duplicates('case_id').lumis_class, imgs.drop_duplicates('case_id').split, margins=True))
print('\nPhotos per split (training uses all photos):')
print(pd.crosstab(imgs.lumis_class, imgs.split, margins=True))
assert imgs.groupby('group').split.nunique().max() == 1

## 3. Save (frozen)

In [ ]:
with zipfile.ZipFile('/content/scin_images.zip', 'w', zipfile.ZIP_STORED) as z:
    for p in imgs.path:
        z.write(f'/content/{p}', p)
import shutil
shutil.copy('/content/scin_images.zip', OUT_ZIP)
imgs.drop(columns=['group']).to_csv(OUT_CSV, index=False)
print('Saved', OUT_CSV, 'and', OUT_ZIP, f'({os.path.getsize(OUT_ZIP) / 1e6:.0f} MB)')